# FOMO (Edge Impulse) → desviación → bits de dirección para el microcontrolador

**Objetivo:** detectar un objeto con un modelo **FOMO** de Edge Impulse, medir cuánto se desvía del **centro de la cámara** y convertirlo en un **byte de flags** (izquierda/derecha/arriba/abajo/centrado) que un microcontrolador pueda usar para mover un motor.

```
 Cámara USB ──► Linux (MPU)  ─────────────────────────────►  STM32 (MCU)
                Python: .eim FOMO → centroide → desviación     sketch: lee el byte
                        → byte de flags ── Bridge RPC ───────►  y mueve el stepper
```
En el **Arduino UNO Q** la parte de Python corre en el Qualcomm QRB2210 (Debian) y el sketch en el STM32U585 (Zephyr). Se comunican con **Bridge** (RPC): Python hace `Bridge.call("set_flags", byte)` y el sketch lo recibe con `Bridge.provide_safe("set_flags", handler)`. En la laptop se usa el mismo código y se imprime el byte o se manda por serial a un Arduino clásico.

## ¿Qué devuelve FOMO?
FOMO (*Faster Objects, More Objects*) **no** es un detector de cajas como YOLO o SSD:
- Se entrena con el **centroide** de cada caja etiquetada y predice una **rejilla de probabilidades** de 1/8 de la resolución de entrada (96×96 → 12×12 celdas).
- En inferencia solo da **dónde** está el objeto, **no su tamaño**.
- El SDK fusiona las celdas activas vecinas y las devuelve en `bounding_boxes` (`x, y, width, height, label, value`). Ese ancho y alto son celdas de la rejilla, no el tamaño real del objeto.
- Por eso usamos **solo el centro** de cada caja fusionada: `centroide = (x + w/2, y + h/2)`.

Fuentes: [Edge Impulse – Detect objects with centroids (FOMO)](https://docs.edgeimpulse.com/tutorials/end-to-end/object-detection-centroids.md) · [Foro EI: FOMO usa centroides](https://forum.edgeimpulse.com/t/fomo-using-center-point-detection-and-not-bounding-boxes/14333) · [UNO Q User Manual (Bridge)](https://docs.arduino.cc/tutorials/uno-q/user-manual/) · [Arduino_RouterBridge](https://github.com/arduino-libraries/Arduino_RouterBridge)

> El código de este notebook es el mismo que `uno_q_app/python/fomo_core.py`. Si cambias algo aquí, cópialo allá (o al revés) para que la app del UNO Q se comporte igual.

## 1. Imports

In [ ]:
import math
import os
import platform
import stat
import sys
import time
import types
from dataclasses import dataclass

import cv2
import numpy as np

# edge_impulse_linux importa su módulo de audio (pyaudio) al cargarse.
# No usamos audio: si pyaudio no está instalado ponemos un módulo vacío.
try:
    import pyaudio  # noqa: F401
except ImportError:
    sys.modules["pyaudio"] = types.ModuleType("pyaudio")

from edge_impulse_linux.image import ImageImpulseRunner

In [ ]:
import matplotlib.pyplot as plt

## 2. Configuración
Todos los parámetros ajustables están en `Config`. Los más importantes para mañana:
- `deadzone`: qué tan lejos del centro tiene que estar el objeto para mandar LEFT/RIGHT. Si el motor tiembla en el centro, súbela.
- `hysteresis`: margen extra para *salir* de LEFT/RIGHT y evitar que el motor oscile en el borde de la zona muerta.
- `smoothing` y `lost_hold_frames`: estabilidad cuando FOMO parpadea.
- `RUN_CAMERA = False` deja correr todo el notebook sin cámara (pruebas con datos sintéticos).

In [ ]:
@dataclass
class Config:
    model_path: str = "modelo.eim"
    camera_id: int = 0
    min_confidence: float = 0.5   # descarta centroides con menor confianza
    target_label: str | None = None  # None = cualquier etiqueta
    target_strategy: str = "closest_prev"  # "confidence" | "closest_center" | "closest_prev"
    deadzone: float = 0.10        # fracción de media imagen (0.10 = ±10 % alrededor del centro)
    hysteresis: float = 0.03      # margen para salir de LEFT/RIGHT/UP/DOWN (evita oscilar)
    smoothing: float = 0.5        # EMA: peso del centroide nuevo (1.0 = sin suavizado)
    lost_hold_frames: int = 5     # frames que se mantiene la última posición si se pierde
    hfov_deg: float = 60.0        # campo de visión horizontal de la cámara
    vfov_deg: float = 45.0        # campo de visión vertical

In [ ]:
cfg = Config(
    model_path="cesar727-project-1-linux-x86_64-v2-fomo.eim",  # ejemplo: carga, pero no está entrenado
    camera_id=0,
    min_confidence=0.5,
    target_label=None,           # p. ej. "audifono"
    target_strategy="closest_prev",
    deadzone=0.10,
    hysteresis=0.03,
    smoothing=0.5,
    lost_hold_frames=5,
    hfov_deg=60.0,
    vfov_deg=45.0,
)
RUN_CAMERA = False   # pon True para las celdas que usan la cámara

## 3. Cargar el modelo `.eim`
`load_model`:
1. Lee la cabecera ELF del `.eim` y verifica que sea de la **misma arquitectura** que la máquina. En la laptop se usa x86_64 y en el UNO Q **aarch64**: hay que descargar el build *Linux (AARCH64)*.
2. Le pone permiso de ejecución.
3. Lo inicia e imprime etiquetas, resolución y *resize mode*.
4. Avisa si el modelo no es FOMO (`model_type != "constrained_object_detection"`).

In [ ]:
_ELF_MACHINES = {0x3E: "x86_64", 0xB7: "aarch64", 0x28: "armv7l"}


def eim_architecture(path):
    """Lee la cabecera ELF del .eim y devuelve su arquitectura (x86_64, aarch64...)."""
    with open(path, "rb") as f:
        header = f.read(20)
    if header[:4] != b"\x7fELF":
        raise ValueError(f"{path} no es un ejecutable ELF (¿descarga incompleta?)")
    machine = int.from_bytes(header[18:20], "little")
    return _ELF_MACHINES.get(machine, f"desconocida (0x{machine:x})")


def load_model(path, verbose=True):
    """Carga un .eim de Edge Impulse y devuelve (runner, model_info)."""
    path = os.path.abspath(path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"No existe el modelo: {path}")

    arch, host = eim_architecture(path), platform.machine()
    if arch != host:
        raise RuntimeError(
            f"El .eim es para {arch} pero esta máquina es {host}. "
            f"Descarga el build 'Linux ({host})' en Edge Impulse → Deployment."
        )

    # El .eim debe ser ejecutable
    mode = os.stat(path).st_mode
    if not mode & stat.S_IXUSR:
        os.chmod(path, mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

    runner = ImageImpulseRunner(path)
    info = runner.init()
    p = info["model_parameters"]
    if verbose:
        print(f"Proyecto    : {info['project']['owner']} / {info['project']['name']}")
        print(f"Arquitectura: {arch}")
        print(f"Tipo        : {p['model_type']}")
        print(f"Etiquetas   : {p['labels']}")
        print(f"Entrada     : {p['image_input_width']}x{p['image_input_height']}"
              f" x{p['image_channel_count']} canales, resize '{runner.resizeMode}'")
    if p["model_type"] != "constrained_object_detection":
        print(f"⚠ El modelo no es FOMO (model_type={p['model_type']}): "
              "las cajas sí tendrán tamaño real y el centroide será el centro de la caja.")
    return runner, info

In [ ]:
runner, model_info = load_model(cfg.model_path)

## 4. De coordenadas del modelo a coordenadas de la cámara
El SDK redimensiona el frame al tamaño de entrada del modelo según el *resize mode* de Studio (`squash`, `fit-shortest` o `fit-longest`). Los centroides salen en coordenadas del modelo y hay que regresarlos a píxeles del frame para medir la desviación respecto a **su** centro.

In [ ]:
def model_to_frame(x, y, frame_w, frame_h, model_w, model_h, mode):
    """Convierte (x, y) de coordenadas del modelo a coordenadas del frame original.

    Edge Impulse redimensiona el frame según el 'resize mode' configurado en Studio.
    """
    if mode in ("", "not-reported", "squash"):
        return x * frame_w / model_w, y * frame_h / model_h

    if mode == "fit-shortest":
        # Recorte central con la relación de aspecto del modelo y luego resize
        aspect = model_w / model_h
        if frame_w / frame_h > aspect:      # frame más ancho: se recortan los lados
            crop_w = int(frame_h * aspect)
            off_x, off_y = (frame_w - crop_w) // 2, 0
            scale = frame_h / model_h
        else:                               # frame más alto: se recorta arriba/abajo
            crop_h = int(frame_w / aspect)
            off_x, off_y = 0, (frame_h - crop_h) // 2
            scale = frame_w / model_w
        return off_x + x * scale, off_y + y * scale

    if mode == "fit-longest":
        # Letterbox: se escala manteniendo aspecto y se rellena con bordes negros
        scale = min(model_w / frame_w, model_h / frame_h)
        pad_x = (model_w - int(frame_w * scale)) // 2
        pad_y = (model_h - int(frame_h * scale)) // 2
        return (x - pad_x) / scale, (y - pad_y) / scale

    raise ValueError(f"Resize mode no soportado: {mode}")

## 5. Inferencia y centroides
- `run_inference`: convierte BGR→RGB, prepara los features con la configuración de Studio y clasifica.
- `get_centroids`: toma el centro de cada caja fusionada, filtra por confianza y etiqueta, y lo pasa a coordenadas del frame. A propósito **no** devuelve ancho ni alto (FOMO no los predice).

In [ ]:
def run_inference(runner, frame_bgr):
    """Ejecuta el modelo sobre un frame BGR de OpenCV y devuelve la respuesta cruda."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    features, _ = runner.get_features_from_image_auto_studio_settings(rgb)
    return runner.classify(features)


def get_centroids(res, frame_w, frame_h, model_w, model_h, mode,
                  min_confidence=0.5, target_label=None):
    """Extrae los centroides FOMO en coordenadas del frame.

    FOMO solo predice *dónde* está el objeto (una rejilla 1/8 de la entrada).
    El SDK fusiona las celdas activas en `bounding_boxes`; su centro es el
    centroide. El ancho/alto NO es el tamaño del objeto, así que no se devuelve.
    """
    centroids = []
    for bb in res.get("result", {}).get("bounding_boxes", []):
        if bb["value"] < min_confidence:
            continue
        if target_label is not None and bb["label"] != target_label:
            continue
        cx, cy = model_to_frame(bb["x"] + bb["width"] / 2, bb["y"] + bb["height"] / 2,
                                frame_w, frame_h, model_w, model_h, mode)
        centroids.append({"label": bb["label"], "confidence": bb["value"], "x": cx, "y": cy})
    return centroids

## 6. Elegir un objetivo
FOMO detecta **muchos** objetos a la vez, pero el motor solo puede seguir uno. `closest_prev` es la opción recomendada: se queda con el mismo objeto entre frames en vez de saltar al de mayor confianza.

In [ ]:
def pick_target(centroids, frame_w, frame_h, strategy="confidence", prev=None):
    """Elige un único objetivo cuando FOMO devuelve varios centroides.

    - "confidence": el de mayor confianza.
    - "closest_center": el más cercano al centro de la imagen (menos movimiento).
    - "closest_prev": el más cercano al objetivo anterior (no salta entre objetos).
    """
    if not centroids:
        return None
    if strategy == "closest_prev" and prev is not None:
        ref = prev
    elif strategy in ("closest_center", "closest_prev"):
        ref = (frame_w / 2, frame_h / 2)
    else:
        return max(centroids, key=lambda c: c["confidence"])
    return min(centroids, key=lambda c: (c["x"] - ref[0]) ** 2 + (c["y"] - ref[1]) ** 2)

## 7. Filtro del centroide
- **EMA** (media móvil exponencial): `pos = α·nuevo + (1-α)·anterior`. Menos ruido, un poco más de retraso.
- **Hold**: si el objeto desaparece unos pocos frames, se mantiene la última posición y se marca el flag `HOLD`. Así el motor no frena y arranca cada vez que FOMO parpadea.

In [ ]:
class CentroidFilter:
    """Suaviza el centroide (media móvil exponencial) y tolera pérdidas breves.

    FOMO a veces "parpadea" (pierde el objeto 1-2 frames). En vez de mandar
    "no detectado" al motor, se mantiene la última posición `hold_frames` frames.
    """

    def __init__(self, smoothing=0.5, hold_frames=5):
        self.alpha = smoothing
        self.hold_frames = hold_frames
        self.pos = None
        self.missed = 0

    def update(self, target):
        """Devuelve ((x, y), held) o (None, False) si el objetivo se perdió."""
        if target is not None:
            new = (target["x"], target["y"])
            if self.pos is None or self.missed > 0:
                self.pos = new
            else:
                a = self.alpha
                self.pos = (a * new[0] + (1 - a) * self.pos[0],
                            a * new[1] + (1 - a) * self.pos[1])
            self.missed = 0
            return self.pos, False

        if self.pos is not None and self.missed < self.hold_frames:
            self.missed += 1
            return self.pos, True

        self.pos = None
        return None, False

## 8. Desviación respecto al centro de la cámara
Eje central = mitad de la imagen `(W/2, H/2)`. Punto de referencia = centroide FOMO (ya filtrado).

$$dx = c_x - \tfrac{W}{2} \qquad dy = \tfrac{H}{2} - c_y \qquad dx_{norm} = \frac{dx}{W/2}\in[-1,1] \qquad \theta_x = \arctan\!\big(dx_{norm}\cdot\tan(\tfrac{HFOV}{2})\big)$$

`dx > 0` → derecha; `dy > 0` → arriba. Los grados sirven si quieres convertir a pasos del motor (2048 pasos/vuelta → 5.69 pasos/°).

In [ ]:
def compute_deviation(centroid, frame_w, frame_h, hfov_deg=60.0, vfov_deg=45.0):
    """Desviación del centroide respecto al centro de la cámara.

    Ejes: x positivo a la derecha, y positivo hacia arriba.
    """
    cx, cy = centroid
    center_x, center_y = frame_w / 2, frame_h / 2

    dx_px = cx - center_x
    dy_px = center_y - cy           # invertimos y: en la imagen crece hacia abajo

    dx_norm = dx_px / center_x      # [-1, 1]
    dy_norm = dy_px / center_y

    # Modelo pinhole: ángulo = atan(desplazamiento_normalizado * tan(FOV/2))
    dx_deg = math.degrees(math.atan(dx_norm * math.tan(math.radians(hfov_deg / 2))))
    dy_deg = math.degrees(math.atan(dy_norm * math.tan(math.radians(vfov_deg / 2))))

    return {"dx_px": dx_px, "dy_px": dy_px,
            "dx_norm": dx_norm, "dy_norm": dy_norm,
            "dx_deg": dx_deg, "dy_deg": dy_deg}

## 9. Codificación binaria (1 byte para el MCU)
| bit | valor | flag | significado |
|---|---|---|---|
| 0 | `0x01` | `DETECTED` | hay objetivo |
| 1 | `0x02` | `LEFT` | objeto a la izquierda → girar a la izquierda |
| 2 | `0x04` | `RIGHT` | objeto a la derecha |
| 3 | `0x08` | `UP` | objeto arriba |
| 4 | `0x10` | `DOWN` | objeto abajo |
| 5 | `0x20` | `CENTERED` | dentro de la zona muerta en x **y** y |
| 6 | `0x40` | `HOLD` | posición estimada (objeto perdido momentáneamente) |
| 7 | `0x80` | — | libre (p. ej. un botón o un modo) |

Ejemplos: `0b00000000` = nada detectado (motor quieto) · `0b00100001` = centrado · `0b00001011` = izquierda y arriba.

**Histéresis:** para pasar a LEFT o RIGHT hay que superar `deadzone`, y para volver al centro hay que bajar de `deadzone - hysteresis`. Sin esto, un objeto justo en el borde haría que el motor fuera y viniera sin parar.

In [ ]:
DETECTED = 1 << 0   # hay objetivo
LEFT     = 1 << 1   # objeto a la izquierda del centro
RIGHT    = 1 << 2   # objeto a la derecha
UP       = 1 << 3   # objeto arriba
DOWN     = 1 << 4   # objeto abajo
CENTERED = 1 << 5   # dentro de la zona muerta en ambos ejes
HOLD     = 1 << 6   # posición estimada (objeto perdido, se usa la última conocida)

FLAG_NAMES = {DETECTED: "DETECTED", LEFT: "LEFT", RIGHT: "RIGHT", UP: "UP",
              DOWN: "DOWN", CENTERED: "CENTERED", HOLD: "HOLD"}


def _axis_state(value, state, deadzone, hysteresis):
    """-1, 0 o +1 con histéresis: se entra a ±1 al pasar `deadzone` y se vuelve
    a 0 solo al bajar de `deadzone - hysteresis`."""
    if abs(value) > deadzone:
        return 1 if value > 0 else -1
    if state != 0 and abs(value) > deadzone - hysteresis:
        return state
    return 0


class DirectionEncoder:
    """Convierte la desviación en un byte de flags para el microcontrolador."""

    def __init__(self, deadzone=0.10, hysteresis=0.03):
        self.deadzone = deadzone
        self.hysteresis = hysteresis
        self.sx = 0
        self.sy = 0

    def encode(self, deviation, held=False):
        if deviation is None:
            self.sx = self.sy = 0
            return 0
        self.sx = _axis_state(deviation["dx_norm"], self.sx, self.deadzone, self.hysteresis)
        self.sy = _axis_state(deviation["dy_norm"], self.sy, self.deadzone, self.hysteresis)

        flags = DETECTED
        flags |= LEFT if self.sx < 0 else RIGHT if self.sx > 0 else 0
        flags |= DOWN if self.sy < 0 else UP if self.sy > 0 else 0
        if self.sx == 0 and self.sy == 0:
            flags |= CENTERED
        if held:
            flags |= HOLD
        return flags


def decode_flags(flags):
    """Texto legible de un byte de flags, p. ej. 'DETECTED|LEFT|UP'."""
    names = [n for bit, n in FLAG_NAMES.items() if flags & bit]
    return "|".join(names) if names else "NONE"

## 10. Pipeline completo
`Pipeline.process(frame)` hace todo para un frame y devuelve un diccionario con `centroids`, `target`, `pos`, `held`, `deviation`, `flags`, `timing` e `infer_s`. `process_result` hace lo mismo a partir de una respuesta ya calculada, para probar sin cámara ni modelo entrenado.

In [ ]:
class Pipeline:
    """Todo el proceso para un frame: inferencia → centroide → filtro → desviación → flags."""

    def __init__(self, runner, cfg: Config):
        self.runner = runner
        self.cfg = cfg
        self.model_w, self.model_h = runner.dim
        self.filter = CentroidFilter(cfg.smoothing, cfg.lost_hold_frames)
        self.encoder = DirectionEncoder(cfg.deadzone, cfg.hysteresis)

    def process(self, frame_bgr):
        h, w = frame_bgr.shape[:2]
        t0 = time.perf_counter()
        res = run_inference(self.runner, frame_bgr)
        return self.process_result(res, w, h, time.perf_counter() - t0)

    def process_result(self, res, w, h, infer_s=0.0):
        """Igual que process() pero a partir de una respuesta ya calculada (útil para pruebas)."""
        c = self.cfg
        self.encoder.deadzone, self.encoder.hysteresis = c.deadzone, c.hysteresis
        centroids = get_centroids(res, w, h, self.model_w, self.model_h, self.runner.resizeMode,
                                  c.min_confidence, c.target_label)
        target = pick_target(centroids, w, h, c.target_strategy, self.filter.pos)
        pos, held = self.filter.update(target)
        deviation = compute_deviation(pos, w, h, c.hfov_deg, c.vfov_deg) if pos else None
        flags = self.encoder.encode(deviation, held)
        return {"centroids": centroids, "target": target, "pos": pos, "held": held,
                "deviation": deviation, "flags": flags,
                "timing": res.get("timing", {}), "infer_s": infer_s}

In [ ]:
pipeline = Pipeline(runner, cfg)

## 11. Envío al microcontrolador
`FlagSender` manda el byte **solo cuando cambia**, más un **latido** cada 200 ms. Con eso el bus no se satura y el MCU puede usar un *timeout* (500 ms en el sketch) para detener el motor si Python se cae.

| Transporte | Dónde | Lado MCU |
|---|---|---|
| `PrintTransport` | laptop, depuración | — |
| `SerialTransport` | laptop + Arduino clásico por USB | `arduino_serial_flags/arduino_serial_flags.ino` |
| `BridgeTransport` | **Arduino UNO Q** (App Lab) | `uno_q_app/sketch/sketch.ino` |

En el UNO Q el handler del sketch está registrado con `Bridge.provide_safe` y **solo guarda el byte**. La documentación advierte que llamar `Bridge.call` o imprimir dentro de un handler puede causar un *deadlock*.

In [ ]:
class FlagSender:
    """Envía el byte solo cuando cambia, más un 'latido' periódico.

    El latido permite que el MCU detenga el motor si deja de recibir datos
    (p. ej. si se cae la app de Python).
    """

    def __init__(self, transport, heartbeat_s=0.2):
        self.transport = transport
        self.heartbeat_s = heartbeat_s
        self.last = None
        self.last_t = 0.0

    def send(self, flags):
        now = time.monotonic()
        if flags != self.last or now - self.last_t >= self.heartbeat_s:
            self.transport.write(flags)
            self.last, self.last_t = flags, now
            return True
        return False


class PrintTransport:
    """Para depurar: imprime el byte en vez de enviarlo."""

    def write(self, flags):
        print(f"→ MCU 0b{flags:08b} ({decode_flags(flags)})")


class SerialTransport:
    """Arduino clásico por USB: envía 1 byte crudo (ver arduino_serial_flags.ino)."""

    def __init__(self, port="/dev/ttyACM0", baud=115200):
        import serial
        self.ser = serial.Serial(port, baud, timeout=0)
        time.sleep(2)  # el Arduino se reinicia al abrir el puerto

    def write(self, flags):
        self.ser.write(bytes([flags]))

    def close(self):
        self.ser.close()


class BridgeTransport:
    """Arduino UNO Q: llama a `set_flags` en el sketch del STM32 vía Bridge (RPC)."""

    def __init__(self, method="set_flags"):
        from arduino.app_utils import Bridge
        self.bridge = Bridge
        self.method = method

    def write(self, flags):
        self.bridge.call(self.method, int(flags))

## 12. Visualización

In [ ]:
def draw_overlay(frame, out, cfg: Config, fps=None):
    """Dibuja ejes, zona muerta, centroides, objetivo, desviación y el byte enviado."""
    img = frame.copy()
    h, w = img.shape[:2]
    cx, cy = w // 2, h // 2

    # Ejes centrales y zona muerta
    cv2.line(img, (cx, 0), (cx, h), (200, 200, 200), 1)
    cv2.line(img, (0, cy), (w, cy), (200, 200, 200), 1)
    dzx, dzy = int(cfg.deadzone * w / 2), int(cfg.deadzone * h / 2)
    cv2.rectangle(img, (cx - dzx, cy - dzy), (cx + dzx, cy + dzy), (255, 255, 255), 1)

    for c in out["centroids"]:
        p = (int(c["x"]), int(c["y"]))
        cv2.circle(img, p, 6, (0, 255, 0), 2)
        cv2.putText(img, f"{c['label']} {c['confidence']:.2f}", (p[0] + 8, p[1] - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    flags, dev = out["flags"], out["deviation"]
    if out["pos"] is not None:
        p = (int(out["pos"][0]), int(out["pos"][1]))
        color = (0, 165, 255) if out["held"] else (0, 0, 255)
        cv2.circle(img, p, 8, color, -1)
        cv2.arrowedLine(img, (cx, cy), p, (0, 255, 255), 2, tipLength=0.05)
        cv2.putText(img, f"dx={dev['dx_px']:+.0f}px ({dev['dx_deg']:+.1f} deg)  "
                         f"dy={dev['dy_px']:+.0f}px ({dev['dy_deg']:+.1f} deg)",
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    else:
        cv2.putText(img, "Sin deteccion", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    cv2.putText(img, f"0b{flags:08b}  {decode_flags(flags)}", (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    if fps is not None:
        cv2.putText(img, f"{fps:.1f} FPS", (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return img

## 13. Pruebas sin cámara
El modelo de ejemplo carga pero no está entrenado, así que:
1. Se corre una inferencia **real** sobre un frame negro para comprobar que el `.eim` responde.
2. Se prueba el resto del pipeline con respuestas **sintéticas** que tienen el mismo formato que devuelve FOMO.

In [ ]:
black = np.zeros((480, 640, 3), np.uint8)
res = run_inference(runner, black)
assert "result" in res and "bounding_boxes" in res["result"], res
print("Respuesta real del .eim:", res)

In [ ]:
W, H = 640, 480
MW, MH = pipeline.model_w, pipeline.model_h

def fake_fomo(*points, label=None, conf=0.9, cell=8):
    """Respuesta con el formato de FOMO: cajas de 1 celda centradas en puntos (x, y) del frame (squash)."""
    label = label or model_info["model_parameters"]["labels"][0]
    boxes = []
    for x, y in points:
        mx, my = x * MW / W, y * MH / H
        boxes.append({"label": label, "value": conf, "x": mx - cell / 2, "y": my - cell / 2,
                      "width": cell, "height": cell})
    return {"result": {"bounding_boxes": boxes}}

def run(res):
    return pipeline.process_result(res, W, H)

pipeline = Pipeline(runner, cfg)      # estado limpio
cases = [
    ((320, 240), DETECTED | CENTERED),
    ((100, 240), DETECTED | LEFT),
    ((600, 50),  DETECTED | RIGHT | UP),
    ((320, 470), DETECTED | DOWN),
]
for point, expected in cases:
    pipeline = Pipeline(runner, cfg)
    out = run(fake_fomo(point))
    print(f"{str(point):12s} → 0b{out['flags']:08b}  {decode_flags(out['flags']):24s} "
          f"dx={out['deviation']['dx_deg']:+6.1f}°  dy={out['deviation']['dy_deg']:+6.1f}°")
    assert out["flags"] == expected, (point, decode_flags(out["flags"]))

# Confianza baja → se ignora
pipeline = Pipeline(runner, cfg)
assert run(fake_fomo((100, 240), conf=0.2))["flags"] == 0

# Pérdida breve → HOLD, pérdida larga → 0
pipeline = Pipeline(runner, cfg)
run(fake_fomo((100, 240)))
empty = {"result": {"bounding_boxes": []}}
held = [run(empty)["flags"] for _ in range(cfg.lost_hold_frames + 1)]
assert all(f & HOLD for f in held[:-1]) and held[-1] == 0, [decode_flags(f) for f in held]

# Varios objetos con closest_prev: se queda con el que venía siguiendo
pipeline = Pipeline(runner, cfg)
run(fake_fomo((150, 240)))
out = run(fake_fomo((160, 240), (600, 240), conf=0.9))
assert out["flags"] & LEFT, decode_flags(out["flags"])
print("✓ todas las pruebas pasaron")

**Histéresis en acción:** el objeto barre de izquierda a derecha y regresa. Fíjate en que el cambio a RIGHT y la vuelta al centro ocurren en posiciones distintas.

In [ ]:
pipeline = Pipeline(runner, cfg)
xs = np.concatenate([np.linspace(320, 420, 21), np.linspace(420, 320, 21)])
states = []
for x in xs:
    f = run(fake_fomo((x, 240)))["flags"]
    states.append(1 if f & RIGHT else 0)

dx_norm = (xs - W / 2) / (W / 2)
plt.figure(figsize=(8, 2.5))
plt.plot(dx_norm[:21], states[:21], "o-", label="objeto alejándose")
plt.plot(dx_norm[21:], states[21:], "s--", label="objeto regresando")
plt.axvline(cfg.deadzone, color="gray", ls=":", label="deadzone")
plt.axvline(cfg.deadzone - cfg.hysteresis, color="gray", ls="-.", label="deadzone - hysteresis")
plt.yticks([0, 1], ["CENTRO", "RIGHT"]); plt.xlabel("dx_norm"); plt.legend(fontsize=8); plt.tight_layout()
plt.show()

## 14. Un solo frame de la cámara

In [ ]:
def snapshot(cfg):
    cam = cv2.VideoCapture(cfg.camera_id)
    ok, frame = cam.read()
    cam.release()
    assert ok, "No se pudo leer de la cámara"
    out = Pipeline(runner, cfg).process(frame)
    print(f"Centroides: {len(out['centroids'])}  flags=0b{out['flags']:08b} ({decode_flags(out['flags'])})")
    if out["deviation"]:
        for k, v in out["deviation"].items():
            print(f"  {k:8s} = {v:+.3f}")
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(draw_overlay(frame, out, cfg), cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

if RUN_CAMERA:
    snapshot(cfg)

## 15. Tiempo real
Elige el transporte:
- `PrintTransport()` solo imprime.
- `SerialTransport("/dev/ttyACM0")` manda el byte a un Arduino con `arduino_serial_flags.ino`.

Presiona `q` en la ventana para salir.

In [ ]:
def live(cfg, transport=None):
    pipe = Pipeline(runner, cfg)
    sender = FlagSender(transport or PrintTransport(), heartbeat_s=0.2)
    cam = cv2.VideoCapture(cfg.camera_id)
    t_prev, fps = time.perf_counter(), 0.0
    try:
        while True:
            ok, frame = cam.read()
            if not ok:
                break
            out = pipe.process(frame)
            sender.send(out["flags"])
            now = time.perf_counter()
            fps = 0.9 * fps + 0.1 / max(now - t_prev, 1e-6)
            t_prev = now
            cv2.imshow("FOMO tracker", draw_overlay(frame, out, cfg, fps))
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        sender.transport.write(0)   # deja el motor quieto al salir
        cam.release()
        cv2.destroyAllWindows()

if RUN_CAMERA:
    live(cfg, PrintTransport())
    # live(cfg, SerialTransport("/dev/ttyACM0"))

---
# 🛠 Funcionalidades útiles para el hackatón

| # | Utilidad | Para qué | ¿Necesita cámara? |
|---|---|---|---|
| A | `benchmark` | Medir FPS y latencia reales y decidir la resolución del modelo | No |
| B | `set_model_threshold` | Cambiar el umbral de confianza *dentro* del modelo sin reentrenar | No |
| C | `simulate_mcu` | Ver qué haría el motor con cada byte, sin hardware | No |
| D | `steps_for_angle` | Pasar de grados de desviación a pasos del stepper (control proporcional) | No |
| E | `collect_dataset` | Capturar fotos del lugar real para reentrenar en Edge Impulse | Sí |
| F | `calibrate_live` | Ajustar zona muerta y confianza con sliders mientras ves la cámara | Sí |
| G | `record_demo` | Grabar un video con el overlay para la presentación | Sí |
| H | Llevarlo al UNO Q | Pasos para correr `uno_q_app/` en App Lab | — |
| I | Troubleshooting | Errores típicos y cómo resolverlos | — |

### A. Benchmark de latencia
Mide por separado el tiempo de preparar los features en Python, el DSP y la clasificación que reporta el `.eim`, y el total. **El modelo de ejemplo es de 1080×1080**: eso son 1.1 M de features por frame y será lento, sobre todo en el UNO Q. Para seguimiento en tiempo real entrena FOMO a **96×96 o 160×160** (en Studio: *Create impulse → Image data → Image width/height*).

In [ ]:
def benchmark(runner, n=10, size=(640, 480)):
    frame = np.random.randint(0, 255, (size[1], size[0], 3), np.uint8)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    prep, total, dsp, clf = [], [], [], []
    for _ in range(n):
        t0 = time.perf_counter()
        features, _ = runner.get_features_from_image_auto_studio_settings(rgb)
        t1 = time.perf_counter()
        res = runner.classify(features)
        t2 = time.perf_counter()
        prep.append((t1 - t0) * 1000); total.append((t2 - t0) * 1000)
        dsp.append(res["timing"]["dsp"]); clf.append(res["timing"]["classification"])
    print(f"Entrada del modelo: {runner.dim[0]}x{runner.dim[1]}  ({n} frames {size[0]}x{size[1]})")
    print(f"  features (Python): {np.mean(prep):7.1f} ms")
    print(f"  DSP (.eim)       : {np.mean(dsp):7.1f} ms")
    print(f"  clasificación    : {np.mean(clf):7.1f} ms")
    print(f"  total            : {np.mean(total):7.1f} ms  → {1000 / np.mean(total):.1f} FPS máx.")

benchmark(runner, n=5)

### B. Umbral de confianza dentro del modelo
`cfg.min_confidence` filtra en Python, pero el `.eim` trae su propio umbral (`min_score`, 0.5 por defecto) y **descarta antes** todo lo que esté por debajo. Si tu modelo detecta poco, baja el umbral del `.eim`; si da falsos positivos, súbelo.

In [ ]:
def set_model_threshold(runner, model_info, min_score):
    for th in model_info["model_parameters"].get("thresholds", []):
        if th["type"] == "object_detection":
            runner.set_threshold({"id": th["id"], "min_score": float(min_score)})
            print(f"Umbral del bloque {th['id']} → {min_score}")
            return
    print("Este modelo no expone un umbral configurable")

set_model_threshold(runner, model_info, 0.5)

### C. Simulador del microcontrolador
Reproduce en Python la misma lógica que `sketch.ino`. Sirve para revisar el protocolo antes de tener el motor conectado, o para explicarlo a los jueces.

In [ ]:
def simulate_mcu(flags):
    """Qué hace sketch.ino con este byte."""
    if not flags & DETECTED:
        return "motor detenido (sin objetivo)"
    if flags & CENTERED:
        return "motor detenido (centrado) ✓"
    giro = "⟲ izquierda" if flags & LEFT else "⟳ derecha" if flags & RIGHT else "sin giro (solo desviación vertical)"
    extra = " [estimado]" if flags & HOLD else ""
    return f"girando {giro}{extra}"

pipeline = Pipeline(runner, cfg)
trayectoria = [(80, 240), (200, 240), (300, 240), None, None, (320, 230), None, None, None, None, None, None]
for p in trayectoria:
    out = run(fake_fomo(p) if p else {"result": {"bounding_boxes": []}})
    print(f"{str(p):12s} 0b{out['flags']:08b}  {simulate_mcu(out['flags'])}")

### D. De grados a pasos del motor
Si en lugar de "girar mientras esté a la izquierda" quieres **un solo movimiento proporcional**, convierte la desviación en pasos. Con un 28BYJ-48 (2048 pasos/vuelta) y el motor girando la cámara en el eje horizontal, 1° ≈ 5.7 pasos. Para mandarlo por el bus se puede usar el bit 7 libre como modo o un segundo `Bridge.call("move_steps", n)`.

In [ ]:
def steps_for_angle(deg, steps_per_rev=2048, gear_ratio=1.0):
    return int(round(deg / 360.0 * steps_per_rev * gear_ratio))

for d in (-20, -5, 0, 3, 15):
    print(f"{d:+4d}° → {steps_for_angle(d):+5d} pasos")

### E. Recolector de dataset
Lo que más mejora un modelo FOMO es entrenarlo con fotos **del lugar real** (luz, fondo y distancia de la demo). Presiona `s` para guardar un frame y `q` para salir. Después súbelas a Edge Impulse:
```bash
edge-impulse-uploader --category split dataset/audifono/*.jpg
```
y etiqueta las cajas en *Data acquisition → Labeling queue*. FOMO funciona mejor con objetos de tamaño parecido y que no estén pegados entre sí.

In [ ]:
def collect_dataset(cfg, label="audifono", out_dir="dataset"):
    folder = os.path.join(out_dir, label)
    os.makedirs(folder, exist_ok=True)
    cam = cv2.VideoCapture(cfg.camera_id)
    n = len(os.listdir(folder))
    try:
        while True:
            ok, frame = cam.read()
            if not ok:
                break
            view = frame.copy()
            cv2.putText(view, f"{label}: {n} fotos  [s]=guardar [q]=salir", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            cv2.imshow("dataset", view)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("s"):
                path = os.path.join(folder, f"{label}_{int(time.time() * 1000)}.jpg")
                cv2.imwrite(path, frame)
                n += 1
            elif key == ord("q"):
                break
    finally:
        cam.release()
        cv2.destroyAllWindows()
    print(f"{n} imágenes en {folder}")

if RUN_CAMERA:
    collect_dataset(cfg, label="audifono")

### F. Calibración en vivo
Mueve los sliders mientras apuntas al objeto. Al salir con `q` se imprimen los valores para copiarlos a `Config` y a `uno_q_app/python/main.py`.

In [ ]:
def calibrate_live(cfg):
    win = "calibracion"
    cv2.namedWindow(win)
    cv2.createTrackbar("deadzone %", win, int(cfg.deadzone * 100), 50, lambda v: None)
    cv2.createTrackbar("hysteresis %", win, int(cfg.hysteresis * 100), 20, lambda v: None)
    cv2.createTrackbar("min_conf %", win, int(cfg.min_confidence * 100), 100, lambda v: None)
    cv2.createTrackbar("smoothing %", win, int(cfg.smoothing * 100), 100, lambda v: None)
    pipe = Pipeline(runner, cfg)
    cam = cv2.VideoCapture(cfg.camera_id)
    try:
        while True:
            ok, frame = cam.read()
            if not ok:
                break
            cfg.deadzone = max(cv2.getTrackbarPos("deadzone %", win), 1) / 100
            cfg.hysteresis = min(cv2.getTrackbarPos("hysteresis %", win) / 100, cfg.deadzone)
            cfg.min_confidence = cv2.getTrackbarPos("min_conf %", win) / 100
            pipe.filter.alpha = cfg.smoothing = max(cv2.getTrackbarPos("smoothing %", win), 1) / 100
            out = pipe.process(frame)
            cv2.putText(frame, simulate_mcu(out["flags"]), (10, 75),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
            cv2.imshow(win, draw_overlay(frame, out, cfg))
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cam.release()
        cv2.destroyAllWindows()
    print(f"deadzone={cfg.deadzone:.2f}, hysteresis={cfg.hysteresis:.2f}, "
          f"min_confidence={cfg.min_confidence:.2f}, smoothing={cfg.smoothing:.2f}")

if RUN_CAMERA:
    calibrate_live(cfg)

### G. Grabar la demo
Graba lo que ve el modelo con el overlay (centroides, flechas, byte enviado) en un `.mp4`. Es útil como respaldo si la demo en vivo falla.

In [ ]:
def record_demo(cfg, path="demo_fomo.mp4", seconds=20, fps=15):
    pipe = Pipeline(runner, cfg)
    cam = cv2.VideoCapture(cfg.camera_id)
    ok, frame = cam.read()
    assert ok, "No se pudo leer de la cámara"
    h, w = frame.shape[:2]
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    t_end = time.time() + seconds
    try:
        while time.time() < t_end:
            ok, frame = cam.read()
            if not ok:
                break
            img = draw_overlay(frame, pipe.process(frame), cfg)
            writer.write(img)
            cv2.imshow("grabando", img)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        writer.release()
        cam.release()
        cv2.destroyAllWindows()
    print(f"Video guardado en {os.path.abspath(path)}")

if RUN_CAMERA:
    record_demo(cfg, seconds=20)

### H. Llevarlo al Arduino UNO Q
La carpeta `uno_q_app/` es una app de App Lab lista para copiar:
```
uno_q_app/
├── app.yaml
├── python/main.py         App.run(user_loop=loop) → Bridge.call("set_flags", byte)
├── python/fomo_core.py    el mismo código de este notebook
├── python/model.eim       ← tu modelo, build "Linux (AARCH64)"
└── sketch/sketch.ino      Bridge.provide_safe("set_flags", ...) + stepper no bloqueante
```
1. Edge Impulse → *Deployment* → **Linux (AARCH64)** → descarga y renombra a `python/model.eim`.
2. En App Lab crea una app y copia `python/` y `sketch/`.
3. Conecta la cámara USB (hub USB-C con alimentación) y el ULN2003 a los pines 8, 9, 10 y 11.
4. *Run*. En la consola verás `DETECTED|LEFT  dx=-12.3°`.

**Probar `main.py` en la laptop antes:**
```bash
cd practica/uno_q_app/python
EI_MODEL=../../cesar727-project-1-linux-x86_64-v2-fomo.eim ../../../.venv/bin/python main.py
FLAGS_SERIAL_PORT=/dev/ttyACM0 EI_MODEL=... ../../../.venv/bin/python main.py   # con Arduino por USB
```
Sin `arduino.app_utils`, `main.py` detecta que no está en el UNO Q y usa `PrintTransport` o `SerialTransport`.

**Alternativa sin Python propio:** App Lab trae el brick `video_object_detection`, que acepta un modelo de Edge Impulse (`EI_OBJ_DETECTION_MODEL` en `app.yaml`). Úsalo si en la placa no se puede abrir la cámara con OpenCV o instalar `edge_impulse_linux`.

### I. Troubleshooting
| Síntoma | Causa probable | Solución |
|---|---|---|
| `El .eim es para x86_64 pero esta máquina es aarch64` | Modelo de la laptop en el UNO Q | Descargar el build *Linux (AARCH64)* |
| `PermissionError` al cargar | `.eim` sin permiso de ejecución | `load_model` lo arregla; si no, `chmod +x modelo.eim` |
| `ModuleNotFoundError: pyaudio` | `edge_impulse_linux` importa audio | Ya está resuelto con el stub de la celda de imports |
| Muy pocos FPS | Entrada del modelo grande (1080×1080) | Reentrenar a 96×96 o 160×160 y usar la utilidad A |
| Nunca detecta | Umbral alto o el modelo no generaliza al lugar | Utilidad B, bajar `min_confidence`, recolectar fotos con E |
| El motor tiembla en el centro | Zona muerta chica o sin histéresis | Utilidad F: subir `deadzone` / `hysteresis` / bajar `smoothing` |
| El motor gira al revés | Orden de bobinas o montaje | Invertir el signo de `dir` en el sketch |
| La cámara no abre | Otro proceso la usa o índice incorrecto | Cerrar otras ventanas o notebooks, probar `camera_id=1` |
| El UNO Q se congela con Bridge | `Bridge.call` o `print` dentro de un handler | El handler solo debe guardar el valor (como en `sketch.ino`) |
| El motor arranca solo al encender | El sketch arranca antes que Python | Ya está cubierto: sin mensajes por 500 ms el motor se detiene |

## Liberar recursos

In [ ]:
runner.stop()